In [46]:
import os
import cv2
import time
import torch
import numpy as np
from pathlib import Path

from ultralytics import YOLO

from anomalib.models import Padim
from anomalib.engine import Engine

from PIL import Image
import matplotlib.pyplot as plt

In [47]:
# ===========================
# CONFIGURATION
# ===========================

PROJECT_ROOT = Path.cwd().parent

YOLO_MODEL_PATH = PROJECT_ROOT / "models" / "yolo_screw_detector" / "weights" / "best.pt"

PADIM_CHECKPOINT = PROJECT_ROOT / "checkpoints" / "padim_checkpoint.ckpt"

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

ANOMALY_THRESHOLD = 0.60

CONFIDENCE = 0.40

IMG_SIZE = 640

print("Project Root :", PROJECT_ROOT)
print("Device       :", DEVICE)
print("YOLO Model   :", YOLO_MODEL_PATH)
print("PaDiM Model  :", PADIM_CHECKPOINT)

Project Root : d:\Projects\VisionXM
Device       : cuda
YOLO Model   : d:\Projects\VisionXM\models\yolo_screw_detector\weights\best.pt
PaDiM Model  : d:\Projects\VisionXM\checkpoints\padim_checkpoint.ckpt


In [48]:
print("Loading YOLO detector...")

detector = YOLO(str(YOLO_MODEL_PATH))

print("YOLO loaded successfully.")

Loading YOLO detector...
YOLO loaded successfully.


In [49]:
print("Loading PaDiM checkpoint...")

padim = Padim(
    backbone="resnet18",
    layers=["layer1", "layer2", "layer3"],
    pre_trained=True,
)

ckpt = torch.load(
    PADIM_CHECKPOINT,
    map_location=DEVICE,
)

state_dict = ckpt["state_dict"]

clean_state_dict = {}

for key, value in state_dict.items():

    if key.startswith("model."):

        clean_state_dict[key.replace("model.", "", 1)] = value

    else:

        clean_state_dict[key] = value

missing, unexpected = padim.load_state_dict(
    clean_state_dict,
    strict=False,
)

padim.eval()

print("Checkpoint loaded.")

print("\nMissing Keys:")
print(missing)

print("\nUnexpected Keys:")
print(unexpected)

Loading PaDiM checkpoint...
Checkpoint loaded.

Missing Keys:
['model.idx', 'model.feature_extractor.feature_extractor.conv1.weight', 'model.feature_extractor.feature_extractor.bn1.weight', 'model.feature_extractor.feature_extractor.bn1.bias', 'model.feature_extractor.feature_extractor.bn1.running_mean', 'model.feature_extractor.feature_extractor.bn1.running_var', 'model.feature_extractor.feature_extractor.layer1.0.conv1.weight', 'model.feature_extractor.feature_extractor.layer1.0.bn1.weight', 'model.feature_extractor.feature_extractor.layer1.0.bn1.bias', 'model.feature_extractor.feature_extractor.layer1.0.bn1.running_mean', 'model.feature_extractor.feature_extractor.layer1.0.bn1.running_var', 'model.feature_extractor.feature_extractor.layer1.0.conv2.weight', 'model.feature_extractor.feature_extractor.layer1.0.bn2.weight', 'model.feature_extractor.feature_extractor.layer1.0.bn2.bias', 'model.feature_extractor.feature_extractor.layer1.0.bn2.running_mean', 'model.feature_extractor.featur

In [50]:
print("Creating inference engine...")

engine = Engine()

print("Engine ready.")

Creating inference engine...
Engine ready.


In [51]:
def predict_crop(crop):

    """
    Predict whether a cropped screw
    is Good or Defective.
    """

    temp_file = "temp_crop.png"

    cv2.imwrite(temp_file, crop)

    predictions = engine.predict(
        model=padim,
        data_path=temp_file,
    )

    prediction = predictions[0]

    score = float(prediction.pred_score)

    label = (
        "Defective"
        if score > ANOMALY_THRESHOLD
        else "Good"
    )

    color = (
        (0, 0, 255)
        if label == "Defective"
        else (0, 255, 0)
    )

    if os.path.exists(temp_file):
        os.remove(temp_file)

    return label, score, color

In [57]:
predictions = engine.predict(
    model=padim,
    data_path=r"D:\Projects\VisionXM\data\train\good\007.png"
)

prediction = predictions[0]

print(type(prediction))
print(dir(prediction))

ckpt_path is not provided. Model weights will not be loaded.
The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: Evaluator, ImageVisualizer, PostProcessor, PreProcessor
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


RuntimeError: The size of tensor a (4096) must match the size of tensor b (0) at non-singleton dimension 2

In [41]:
import torch

ckpt = torch.load(PADIM_MODEL, map_location="cpu")

print(type(ckpt))
print(ckpt.keys())

<class 'dict'>
dict_keys(['epoch', 'global_step', 'pytorch-lightning_version', 'state_dict', 'loops', 'callbacks', 'optimizer_states', 'lr_schedulers', 'hparams_name', 'hyper_parameters'])


In [42]:
for key in ckpt["state_dict"].keys():
    print(key)

_is_fitted
post_processor._image_threshold
post_processor._pixel_threshold
post_processor.image_min
post_processor.image_max
post_processor.pixel_min
post_processor.pixel_max
model.idx
model.feature_extractor.feature_extractor.conv1.weight
model.feature_extractor.feature_extractor.bn1.weight
model.feature_extractor.feature_extractor.bn1.bias
model.feature_extractor.feature_extractor.bn1.running_mean
model.feature_extractor.feature_extractor.bn1.running_var
model.feature_extractor.feature_extractor.bn1.num_batches_tracked
model.feature_extractor.feature_extractor.layer1.0.conv1.weight
model.feature_extractor.feature_extractor.layer1.0.bn1.weight
model.feature_extractor.feature_extractor.layer1.0.bn1.bias
model.feature_extractor.feature_extractor.layer1.0.bn1.running_mean
model.feature_extractor.feature_extractor.layer1.0.bn1.running_var
model.feature_extractor.feature_extractor.layer1.0.bn1.num_batches_tracked
model.feature_extractor.feature_extractor.layer1.0.conv2.weight
model.feature

In [38]:
ANOMALY_THRESHOLD = 0.50

In [39]:
def classify_crop(crop):

    with tempfile.NamedTemporaryFile(
        suffix=".png",
        delete=False
    ) as tmp:

        temp_path = tmp.name

    cv2.imwrite(temp_path, crop)

    prediction = inferencer.predict(temp_path)

    score = float(prediction.pred_score)

    if score > ANOMALY_THRESHOLD:
        label = "Defective"
    else:
        label = "Good"

    os.remove(temp_path)

    return label, score

In [40]:
def detect_and_classify(image):

    result = detector(image)[0]

    annotated = image.copy()

    detections = []

    for box in result.boxes:

        x1, y1, x2, y2 = map(int, box.xyxy[0])

        crop = image[y1:y2, x1:x2]

        if crop.size == 0:
            continue

        label, score = classify_crop(crop)

        color = (0,255,0) if label=="Good" else (0,0,255)

        cv2.rectangle(
            annotated,
            (x1,y1),
            (x2,y2),
            color,
            2
        )

        cv2.putText(
            annotated,
            f"{label} {score:.3f}",
            (x1,y1-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            color,
            2
        )

        detections.append({
            "bbox":[x1,y1,x2,y2],
            "label":label,
            "score":score
        })

    return annotated,detections

In [32]:
def detect_and_classify(image):

    output = detector(image)[0]

    annotated = image.copy()
    detections = []

    for box in output.boxes:

        x1, y1, x2, y2 = map(int, box.xyxy[0])

        crop = image[y1:y2, x1:x2]

        if crop.size == 0:
            continue

        label, score = classify_crop(crop)

        color = (0, 255, 0) if label == "Good" else (0, 0, 255)

        cv2.rectangle(
            annotated,
            (x1, y1),
            (x2, y2),
            color,
            2
        )

        cv2.putText(
            annotated,
            f"{label} ({score:.3f})",
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            color,
            2
        )

        detections.append({
            "box": [x1, y1, x2, y2],
            "label": label,
            "score": score
        })

    return annotated, detections

In [ ]:
IMAGE_PATH = r"D:\Projects\VisionXM\data\test\scratch_neck\004.png"

image = cv2.imread(IMAGE_PATH)

result, detections = detect_and_classify(image)

plt.figure(figsize=(10,10))
plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

print(detections)


0: 640x640 1 screw, 24.4ms
Speed: 694.6ms preprocess, 24.4ms inference, 4.1ms postprocess per image at shape (1, 3, 640, 640)


TypeError: Engine.predict() got an unexpected keyword argument 'image'

In [ ]:
cap = cv2.VideoCapture(0)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    result, _ = detect_and_classify(frame)

    cv2.imshow("YOLO + PaDiM", result)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
# Final live inspection runner (YOLO detection + PaDiM good/defective classification)
# Press Q or Esc in the camera window to stop.
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path.cwd().parent
subprocess.run([sys.executable, str(PROJECT_ROOT / 'webcam_screw_inspection.py')], check=True)
